In [ ]:
# We can also use Snowpark for our analyses!
# from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import Session
import pandas as pd
from sklearn.datasets import make_classification
from dotenv import load_dotenv
from snowflake.ml.utils.connection_params import SnowflakeLoginOptions

from snowflake.ml.modeling.model_selection.grid_search_cv import GridSearchCV
from snowflake.ml.modeling.xgboost import XGBClassifier

In [ ]:
load_dotenv()

In [ ]:
session=Session.builder.configs(SnowflakeLoginOptions("my_example_connection")).create()
session

In [ ]:
session.query_tag='classifier-hpo'

In [ ]:
session

In [ ]:
X,y=make_classification(
    n_samples=1000000,
    n_features=6,
    n_informative=2,
    n_redundant=0,
    random_state=0,
    shuffle=True
)

In [ ]:
X=pd.DataFrame(data=X,columns='X1 X2 X3 X4 X5 X6'.split(" "))
y=pd.DataFrame(data=y, columns=["Y"])

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
df=pd.concat(objs=[X,y],axis=1)
df.head()

In [ ]:
df=session.create_dataframe(data=df)
df.show()

In [ ]:
model=GridSearchCV(
    estimator=XGBClassifier(),
    param_grid={
        'n_estimators':[10,20,30,40],
        'learning_rate':[0.01,0.1,0.2]
    },
    cv=5,
    n_jobs=-1,
    verbose=4,
    input_cols='X1 X2 X3 X4 X5 X6'.split(" "),
    output_cols=['PREDICTIONS'],
    label_cols=['Y']
)

In [ ]:
model.fit(dataset=df)

In [ ]:
df.show()

In [ ]:
preds=model.predict(df)

In [ ]:
preds.show()

In [ ]:
preds.select(
    "PREDICTIONS" 
).show()

In [ ]:
model_skl=model.to_sklearn() 

In [ ]:
dir(model_skl)

In [ ]:
model_skl.best_score_

In [ ]:
model_skl.best_params_